In [1]:
%%capture
!pip install langchain>=0.1.17 openai>=1.13.3 langchain_openai>=0.1.6 transformers==4.41.2 datasets>=2.18.0 accelerate>=0.27.2 sentence-transformers>=2.5.1 duckduckgo-search>=5.2.2
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install llama-cpp-python

# 零、总述

随着学习的深入，我们开始使用主要针对**文本生成**进行训练的模型，这类模型通常被称为**生成式预训练的Transformer(Generativ Pre-trained Transformer, GPT)。** 这些模型具有非常强大的能力，可以根据用户提供的提示词生成文本。通过**提示词工程（Prompt Engineering）**，我们可以更合理的设计这些提示词，从而提高模型生成文本的质量。

在本章中，我们将更加深入地探索这些生成式模型，并进一步学习**提示词工程、使用生成式模型进行推理、结果验证，以及对模型输出进行评估** 等内容。

# 一、使用文本生成模型

在开始学习**提示词工程**的基础知识之前，我们首先需要了解如何使用一个文本生成模型。我们应该怎样选择要使用的模型呢？是选择闭源模型（Proprietary）还是开源模型？

这些问题将作为我们使用文本生成模型的起点

## 1.1 选择 Text Generation Model

我们以选择闭源和开源模型来开始选择文本生成模型，即使专有模型的性能会很好，但是为了学习使用，我们使用开源模型。

那么，我们使用 `Phi-3-mini` 模型，它有 3.8B（38亿）的参数量，特别适合在 8G 的 VRAM 上运行。

总的来说，将小模型变成大模型要比从大模型变成小模型要容易得多。更小的模型会提供更好的引导，而且可以为后续大模型的学习会打下坚实的基础

## 1.2 加载文本生成模型

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 加载模型的分词器
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,

)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


还记得我们第一个notebook中的示例么，我们让他讲一个笑话，我们依旧使用这个例子

In [3]:
# 提示词
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Why did the chicken join the band? Because it had the drumsticks!


在内部，`transformers.pipeline` 首先会将我们的 messages 转换成特定的提示词模板，我们可以使用下述的函数看一下模型将我们的 messages 转成了什么：

In [4]:
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

<|user|>
Create a funny joke about chickens.<|end|>
<|endoftext|>


在输出结果中，你可以看到有几个特殊的词元：`<|user|>`、`<|end|>` 和 `<|endoftext|>`，词元的说明在第一章已讲述

## 1.3 控制模型输出

除了提示词工程之外，我们可以调整模型的参数，比如：`temperature` 和 `top_p`.

这些参数会控制输出的随机性。LLM 每次生成一个 token 时，每一个可能的 token 都会被分配一个似然值。

当我们加载模型时，设置 `do_sample=False` 的目的是为了确保它生成的内容有一些连贯性，它意味着每次生成的 token 都是最可能的那一个词。然而，为了使用 `temperature` 和 `top_p` 参数，我们将设置 `do_sample=True`。

1. temperature

`temperature` 参数控制文本生成的随机性和多样性。它定义了有多大的可能会选择小概率的 token。更高的 `temperature` 通常会导致更*多样化*的输出，而更低的 `temperature` 会生成更稳定的输出。请注意，即使你设置了相同的温度值，多次运行下述代码结果也是会变化的，因为 `temperature` 引入了随机选择的行为

In [5]:
# 更高的温度
output1 = pipe(messages, do_sample=True, temperature=0.8)
print(output1[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why did the chicken join a band? Because it had a great set of drumsticks and loved to scratch-tunes!


In [6]:
# 更低的温度
output2 = pipe(messages, do_sample=True, temperature=0.2)
print(output2[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why did the chicken join the band? Because it had the drumsticks!


2. top-p

`top-p`：也叫做**核采样**，它是一种采样的技术，来控制 LLM 来选择哪些 token 的子集，这些 token 的子集就被称为 nucleus(核)。它会选择那些直到累积的概率达到其设定的值的 token，所以，如果将 `top_p` 设置为 1，则它会选择所有 token。`top_p` 对模型生成的 token 的影响如下：

<center>
<img src="./resources/top_p.png">
</center>


In [7]:
output = pipe(messages, do_sample=True, top_p=0.6)
print(output[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why did the chicken join the band? Because it had the drumsticks!


# 二、介绍提示词工程

**提示词工程（Prompt Engineering）** 是我们可以精心设计提示词，可以引导 LLM 生成我们期望的回答，**但是， 提示词工程不仅仅是设计有效的提示词**，它还可以作为一种工具，用来评估模型的输出，以及设计防护机制和安全环节措施。不存在一个完美的提示词设定，未来也可能不会存在。

我们从回答一个问题开始吧：在提示词中应该有什么呢？

## 2.1 提示词的基本成分


LLM 本质上是一台预测机器，我们从基础的示例看起，如下图所示，可以看到由于没有给出明确的指令，因此 LLM 只会尝试续写这个句子。

```text
                         Basic prompt
                 ┌─────────────────────┐
Input ─────────> │    The sky is       │
                 └─────────────────────┘
                           │
                           ▼
                      ┌─────────┐
                      │   LLM   │
                      └─────────┘
                           │
                           ▼
                         Output
                           │
                           ▼
Generated text ─────────> blue.
```

更进一步来看，在提示词工程中，*我们通常会通过提出一个明确的问题，或者指定一个需要 LLM 完成的具体任务来进行设计。* 为了让模型生成我们更期望的回答，我们需要一个结构更加清晰的提示词。如下图所示，可以看到它包含两个组成部分：**指令本身**以及 **与该指令相关的数据**。

```text
                     Instruction prompt

Instruction ──────>  ┌──────────────────────────────────────────────┐
                     │ Classify the text into negative or positive. │
Data ─────────────>  │                                              │
                     │        "This is a great movie!"              │
                     └──────────────────────────────────────────────┘
                                      │
                                      ▼
                                 ┌─────────┐
                                 │   LLM   │
                                 └─────────┘
                                      │
                                      ▼
                                    Output

Generated text ──────>  The text is positive.
```

再进一步来看，我们在上面的基础上添加了 "Text" 和 "Sentiment"，以防止模型生成一个完整的句子。由于 LLM 已经接收到了足够的指令，因此它有能力泛化到我们所提供的结构中，所以我们期望它会输出 `negaitve` 或 `positive`。

```text
                 Instruction prompt
                 With output indicator

Instruction ──────>  ┌──────────────────────────────────────────────┐
                     │ Classify the text into negative or positive. │
                     │                                              │
Output indicators ─> │ Text:      "This is a great movie!"          │ <──── Data
                     │                                              │
                     │ Sentiment:                                   │
                     └──────────────────────────────────────────────┘
                                      │
                                      ▼
                                 ┌─────────┐
                                 │   LLM   │
                                 └─────────┘
                                      │
                                      ▼
                                    Output

Generated text ──────>  Positive
```

我们可以不断增加和调整提示词中的不同元素，直至模型生成我们所期望的回答。

## 2.2 基于指令的提示词

虽然提示词有很多种方式：从讨论哲学到角色扮演，但是往往提示词会被用来回答特定的问题或者完成一项特定的任务，这被称为基于指令的提示词。下图是基于指令的提示词的使用实例：

<center>
<img src="./resources/instruction_based_prompt_use_case.png">
</center>

上面图中的每一个任务都需要不同的提示词格式，不同的指向和不同的发问。下图是几个使用实例的具体例子：

<center>
<img src="./resources/instruction_based_prompt_example.png">
</center>

虽然这些任务需要不同的指令，但是在提示词技术中也有一些重叠的地方来对输出进行改进。这些技术包含但不限于：

1. Specificity: 指向性，明确性

精确地描述你想要完成的。不是去对 LLM 说“给我写一个产品的描述”，而是要说**给我写一个少于两句话的产品描述，要用正式的用语**

2. Hallucination: 幻觉

LLM 可能会堂而皇之地生成不正确的信息，这被称为幻觉。为了减少这个影响，如果 LLM 直到问题的答案的话，我们只让他生成一个。如果他不知道，则让他用 “我不知道” 来代替。

3. Order: 顺序

你的提示词即用指令开始也用指令结束。特别是长提示词，LLM 通常会忽略中间的信息，而去关注开头或者结尾的提示词.

在这三个方面中，可以说**Specificity 是最重要的方面**，通过限制并明确模型应该生成什么内容，可以降低模型生成与实际场景无关内容的概率。就像人与人之间的交流一样，如果没有明确的指令或额外的上下午，就很那判断当前任务究竟要完成什么。


# 三、高级的提示词工程

提示词设计会变得很复杂，接下来，我们将介绍几种用于构建提示词的高级技巧，我们会从迭代式构建复杂提示词的工作流程开始，一直到按顺序使用多个 LLM 来获得更好的结果。最终，我们还会进一步介绍高级推理技术。

## 3.1 提示词潜在的复杂度

一个提示词通常由多个组成部分构成，在最开始的示例中，我们的提示词由**指令、数据和输出指示符组成**，但提示词并不局限于这个三个组成部分，可以根据需要做扩展。

这些高级的组件会让提示词变得相当的复杂。一些常用的组件有：

1. Persona：LLM 应该扮演什么角色
2. Instruction：描述任务本身，尽可能具体
3. Context：用于描述问题或任务背景的额外信息
4. Format：指定 LLM 应该使用什么格式来输出生成的文本
5. Audience：生成文本所面向的目标人群
6. Tone：生成文本的语气
7. Data：数据

下面，我们来扩展之前的分类提示词，并使用前面提到的所有组成部分。我们可以逐步的构建提示词，并探索每一次修改带来的结果。

那我们大致依据下图来进行探索，**注意：不同组件的排列顺序也会影响 LLM 的输出质量，因为其有近因效应(recency effect)和首因效应（primacy effect）**

<center>
<img src="./resources/modular_component_prompt.png">
</center>

In [8]:
# persona = "你是 LLM 的专家，你特别擅长将复杂的论文内容处理成理解简单的总结"
persona = "You are an expert in Large Language models.You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key finding of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researches quickly understand the most vital information of the paper.\n"
# data_format = "请撰写一份要点式摘要，概述该方法，随后用一段简练的文字总结主要结果“
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone = "The tone should be professional and clear.\n"

下面这段文本是 DeepSeek-R1 的论文，其中包括 R1-Zero、纯 RL 推理能力涌现、cold-start、多阶段训练、蒸馏到 1.5B–70B 模型 等论文核心信息。

In [9]:
text = """
DeepSeek-R1 investigates how reinforcement learning can be used to
develop advanced reasoning capabilities in large language models.

The work introduces two reasoning models: DeepSeek-R1-Zero and
DeepSeek-R1. DeepSeek-R1-Zero is trained using large-scale reinforcement
learning without supervised fine-tuning as an initial stage. During
training, the model naturally develops reasoning behaviors such as
self-reflection, verification, and longer reasoning processes. This
demonstrates that sophisticated reasoning abilities can emerge through
reinforcement learning rather than relying entirely on human-written
reasoning examples.

However, DeepSeek-R1-Zero also exhibits several problems, including
poor readability and language mixing. To address these limitations,
DeepSeek-R1 introduces a multi-stage training pipeline that combines
cold-start data, reinforcement learning, and supervised fine-tuning.
The cold-start data helps establish more readable reasoning patterns
before large-scale reinforcement learning is applied.

Experimental evaluations show that DeepSeek-R1 achieves strong
performance on mathematics, coding, and reasoning benchmarks and
reaches performance comparable to leading reasoning models such as
OpenAI o1-1217 on several tasks.

The authors also investigate knowledge distillation. Reasoning patterns
generated by DeepSeek-R1 are transferred to smaller dense models based
on Qwen and Llama architectures. These distilled models range from
1.5B to 70B parameters and demonstrate that reasoning capabilities
learned by a large model can be effectively transferred to smaller
models.

Overall, the study suggests that reinforcement learning can play a
central role in developing reasoning abilities in language models,
while cold-start training and distillation provide practical ways to
improve usability and transfer reasoning capabilities to smaller models.
"""

In [10]:
data = f"Text to summarize: {text}"

下面我们来探索这四种组合，依旧使用 `Phi-3-mini`模型

In [11]:
# 1. instruction + data
print("=============== instruction + data ===============")
query1 = instruction + data
messages = [
    {"role": "user", "content": f"{query1}"}
]
# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])


# 2. persona + instruction + data
print("=============== persona + instruction + data ===============")
query2 = persona + instruction + data
messages = [
    {"role": "user", "content": f"{query2}"}
]
# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])

# 3. context + tone + instruction + data
print("=============== context + tone + instruction + data ===============")
query3 = instruction + data
messages = [
    {"role": "user", "content": f"{query3}"}
]
# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])

# 4. all-in
query4 = persona + instruction + context + data_format + audience + tone + data
messages = [
    {"role": "user", "content": f"{query4}"}
]
# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=============== instruction + data ===============


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The paper DeepSeek-R1 explores the use of reinforcement learning to enhance reasoning in large language models. It introduces two models, DeepSeek-R1-Zero and DeepSeek-R1. DeepSeek-R1-Zero is trained using reinforcement learning without supervised fine-tuning, leading to the development of reasoning behaviors like self-reflection and longer reasoning processes. However, it suffers from poor readability and language mixing. To overcome these issues, DeepSeek-R1 employs a multi-stage training pipeline that includes cold-start data, reinforcement learning, and supervised fine-tuning. This approach results in more readable reasoning patterns.

Experimental results show that DeepSeek-R1 performs well on mathematics, coding, and reasoning benchmarks, achieving comparable performance to leading models like OpenAI o1-1217. The study also examines knowledge distillation, where reasoning patterns from DeepSeek-R1 are transferred to smaller dense models (Qwen and Llama architectures). These disti

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The paper DeepSeek-R1 explores the use of reinforcement learning to enhance reasoning abilities in large language models. It introduces two models, DeepSeek-R1-Zero and DeepSeek-R1. DeepSeek-R1-Zero is trained using reinforcement learning without supervised fine-tuning, leading to the development of reasoning behaviors such as self-reflection and longer reasoning processes. However, it also exhibits issues like poor readability and language mixing. To address these, DeepSeek-R1 employs a multi-stage training pipeline that includes cold-start data, reinforcement learning, and supervised fine-tuning. This approach results in improved readability and reasoning patterns.

Experimental evaluations show that DeepSeek-R1 performs well on various benchmarks, achieving comparable results to leading reasoning models. The study also investigates knowledge distillation, where reasoning patterns from DeepSeek-R1 are transferred to smaller dense models. These distilled models, ranging from 1.5B to 7

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The paper DeepSeek-R1 explores the use of reinforcement learning to enhance reasoning in large language models. It introduces two models, DeepSeek-R1-Zero and DeepSeek-R1. DeepSeek-R1-Zero is trained using reinforcement learning without supervised fine-tuning, leading to the development of reasoning behaviors like self-reflection and longer reasoning processes. However, it suffers from poor readability and language mixing. To overcome these issues, DeepSeek-R1 employs a multi-stage training pipeline that includes cold-start data, reinforcement learning, and supervised fine-tuning. This approach results in more readable reasoning patterns.

Experimental results show that DeepSeek-R1 performs well on mathematics, coding, and reasoning benchmarks, achieving comparable performance to leading models like OpenAI o1-1217. The study also examines knowledge distillation, where reasoning patterns from DeepSeek-R1 are transferred to smaller dense models (Qwen and Llama architectures). These disti

原始 text 翻译如下：

```text
DeepSeek-R1 探讨了如何利用强化学习来培养大语言模型的高级推理能力。

该研究推出了两款推理模型：DeepSeek-R1-Zero 和 DeepSeek-R1。DeepSeek-R1-Zero 采用大规模强化学习进行训练，且未包含监督微调这一初始阶段。在训练过程中，该模型自然地发展出了诸如自我反思、验证以及更长推理链等推理行为。这表明，复杂的推理能力可以通过强化学习涌现，而无需完全依赖人工编写的推理示例。

然而，DeepSeek-R1-Zero 也存在一些问题，例如可读性较差以及语言混杂。为了解决这些局限性，DeepSeek-R1 引入了一个多阶段训练流程，结合了冷启动数据、强化学习和监督微调。冷启动数据有助于在大规模强化学习应用之前，建立起可读性更强的推理模式。

实验评估显示，DeepSeek-R1 在数学、编程和推理基准测试中表现优异，并在多项任务中达到了与 OpenAI o1-1217 等领先推理模型相当的性能水平。

此外，研究人员还探讨了知识蒸馏技术。DeepSeek-R1 生成的推理模式被迁移至基于 Qwen 和 Llama 架构的较小规模稠密模型中。这些蒸馏模型的参数量涵盖 15 亿（1.5B）到 700 亿（70B）不等，结果证明大模型习得的推理能力可以有效地迁移到较小模型上。

总体而言，该研究表明强化学习在培养语言模型推理能力方面发挥着核心作用，而冷启动训练和知识蒸馏则为提升模型易用性及将推理能力迁移至较小模型提供了切实可行的方法。
```


通过不同提示词让生成式模型总结概要的输出结果翻译如下：

1. instruction + data

论文 DeepSeek-R1 探索了利用强化学习来增强大语言模型推理能力的方法。论文介绍了两个模型：DeepSeek-R1-Zero 和 DeepSeek-R1。DeepSeek-R1-Zero 在没有进行监督微调的情况下，仅使用强化学习进行训练，从而涌现出了自我反思以及更长推理过程等推理行为。然而，它也存在可读性较差和语言混杂的问题。为了解决这些问题，DeepSeek-R1 采用了一套多阶段训练流程，其中包括冷启动数据、强化学习和监督微调。这种方法使模型能够生成更加易读的推理过程。

实验结果表明，DeepSeek-R1 在数学、编程和推理基准测试上表现良好，其性能可以与 OpenAI o1-1217 等领先模型相媲美。研究还探讨了知识蒸馏，即将 DeepSeek-R1 的推理模式迁移到更小的稠密模型中，这些模型采用 Qwen 和 Llama 架构。这些蒸馏模型的参数规模从 1.5B 到 70B 不等，结果表明，大模型学习到的推理能力可以有效迁移到更小的模型中。

总之，论文表明，强化学习对于培养语言模型的推理能力至关重要，而冷启动训练和知识蒸馏能够提高模型的可用性，并将推理能力迁移到更小的模型中。

---

2. persona + instruction + data

论文 DeepSeek-R1 探索了利用强化学习来增强大语言模型推理能力的方法。论文介绍了两个模型：DeepSeek-R1-Zero 和 DeepSeek-R1。DeepSeek-R1-Zero 在没有进行监督微调的情况下，仅使用强化学习进行训练，从而涌现出了自我反思和更长推理过程等推理行为。然而，它也存在可读性较差和语言混杂等问题。为了解决这些问题，DeepSeek-R1 采用了一套多阶段训练流程，其中包括冷启动数据、强化学习和监督微调。这种方法改善了模型的可读性和推理模式。

实验评估表明，DeepSeek-R1 在多个基准测试中表现良好，其结果可以与领先的推理模型相媲美。该研究还探讨了知识蒸馏，即将 DeepSeek-R1 的推理模式迁移到更小的稠密模型中。这些蒸馏模型的参数规模从 1.5B 到 70B 不等，结果表明，大模型学习到的推理能力可以有效迁移到更小的模型中。

总之，论文表明，强化学习对于培养语言模型的推理能力至关重要，而冷启动训练和知识蒸馏能够提高模型的可用性，并将推理能力迁移到更小的模型中。

---

3. context + tone + instruction + data

论文 DeepSeek-R1 探索了利用强化学习来增强大语言模型推理能力的方法。论文介绍了两个模型：DeepSeek-R1-Zero 和 DeepSeek-R1。DeepSeek-R1-Zero 在没有进行监督微调的情况下，仅使用强化学习进行训练，从而涌现出了自我反思以及更长推理过程等推理行为。然而，它也存在可读性较差和语言混杂的问题。为了解决这些问题，DeepSeek-R1 采用了一套多阶段训练流程，其中包括冷启动数据、强化学习和监督微调。这种方法使模型能够生成更加易读的推理过程。

实验结果表明，DeepSeek-R1 在数学、编程和推理基准测试上表现良好，其性能可以与 OpenAI o1-1217 等领先模型相媲美。研究还探讨了知识蒸馏，即将 DeepSeek-R1 的推理模式迁移到更小的稠密模型中，这些模型采用 Qwen 和 Llama 架构。这些蒸馏模型的参数规模从 1.5B 到 70B 不等，结果表明，大模型学习到的推理能力可以有效迁移到更小的模型中。

总之，论文表明，强化学习对于培养语言模型的推理能力至关重要，而冷启动训练和知识蒸馏能够提高模型的可用性，并将推理能力迁移到更小的模型中。

* **方法：**

  * DeepSeek-R1-Zero：在没有监督微调的情况下，使用大规模强化学习进行训练。
  * DeepSeek-R1：采用多阶段训练流程，包括冷启动数据、强化学习和监督微调。

* **结果：**

  * DeepSeek-R1-Zero 涌现出了自我反思和验证等推理行为。
  * DeepSeek-R1 通过多阶段训练方法解决了可读性和语言混杂问题。
  * DeepSeek-R1 在各类基准测试中取得了较强的性能，并可以与领先模型相媲美。
  * 知识蒸馏使得推理能力能够迁移到更小的模型中。

* **总结：**
  论文提出了 DeepSeek-R1，这是一种利用强化学习来培养大语言模型高级推理能力的模型。最初的模型 DeepSeek-R1-Zero 在没有监督微调的情况下进行训练，从而涌现出了自我反思和验证等推理行为。然而，它也面临可读性较差和语言混杂等问题。为了解决这些问题，DeepSeek-R1 引入了一套多阶段训练流程，其中包含冷启动数据、强化学习和监督微调，从而改善了模型的可读性和推理模式。该模型在多个基准测试中表现出较强的性能，可以与领先的推理模型相媲美。此外，该研究还探索了知识蒸馏，结果表明，DeepSeek-R1 的推理能力可以有效迁移到参数规模从 1.5B 到 70B 的更小模型中。这项研究强调了强化学习在培养语言模型推理能力方面的潜力，同时也说明了冷启动训练和知识蒸馏在提升模型可用性和推理能力迁移方面的重要作用。


## 3.2 上下文内学习：提供例子

在前面的例子中，我们会尝试准确的描述 LLM 应该做什么。*那我们与其描述任务，为什么不直接展示任务呢？* 那么，我们就可以给大模型提供一些示例，这通常被称为 **in-context learning(上下文内学习)**。你可以不给大模型提供示例，也可以给大模型提供一个示例，也可以提供两个或更多的示例。如下图所示

<center>
<img src="./resources/shot_prompt.png">
</cener>


借用非常经典的一句话，“一个例子胜过千言万语”。我们用一个简单的例子来说明这种方法，这个例子来自于最初介绍该方法的论文。

此提示词的目标是生成一个包含虚构词语的句子，所以为了提高生成句子的质量，会给模型展示一个示例。

我们**必须**通过 `user` 和 `assistant` 来区分例子的问题和答案。如果不这样的话，我们似乎是在和我们自己对话。一般情况下，我们使用 `assistant` 代表答案。


In [12]:
# 虚构词的任务
one_shot_prompt = [
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },
    {
        "role": "assistant",
        "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },
    {
        "role": "user",
        # swing a sword at it： 挥剑砍向它
        "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    }
]

# 分词结果
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

<|user|>
A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:<|end|>
<|assistant|>
I have a Gigamuru that my uncle gave me as a gift. I love to play it at home.<|end|>
<|user|>
To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:<|end|>
<|endoftext|>


In [13]:
output = pipe(one_shot_prompt)
print(output[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


During the medieval reenactment, the knight skillfully screeg the wooden target, impressing the onlookers with his prowess.


## 3.3 提示词链：将问题分解

在前面的示例中，我们探讨了如何将提示词拆分成多个模块化组件，以提升 LLM 的表现，这种方式对于高度复杂的提示词或者任务来说，它可能并不现实。此时我们就可以将一个提示词拆成一条连续的交互链，来分次调用大模型，逐步解决问题。举个例子，假设我们希望让 LLM 根据一系列产品的特征来生成产品名称、宣传口号和销售文案（sales_pitch），那我们就可以首先生成产品名称，然后将产品名称和特征作为输入来生成宣传口号，最后再使用产品特征、产品名称和宣传口号来生成销售文案。

<center>
<img src="./resources/chain_prompt.png">
</center>

In [14]:
# 创建名称和产品名
product_prompt = [
    {"role": "user", "content": "Create a name and slogan for a chatbot that leverages LLMs."}
]
outputs = pipe(product_prompt)
product_description = outputs[0]["generated_text"]
print(product_description)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Name: ChatSage
Slogan: "Your AI Companion for Smart Conversations"


In [15]:
# 创建销售文案
sales_pitch_prompt = [
    {"role": "user", "content": f"Generate a very short sales pitch for the following product: '{product_description}'"}
]

outputs = pipe(sales_pitch_prompt)
print(outputs[0]["generated_text"])

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Introducing ChatSage, your AI Companion for Smart Conversations. With ChatSage, you'll have a personalized and intelligent assistant at your fingertips, ready to engage in meaningful dialogue, provide helpful information, and enhance your daily interactions. Experience the future of communication with ChatSage – your smart conversation partner.


这种方法可以应用于多种使用场景，包括：

1. **Response Validation(响应验证)**

让 LLM 对之前生成的输出进行再次检查

2. **（Parallel Prompts）并行提示**

**并行创建多个提示词，然后在最后一步将它们的结果合并起来。例如，可以让多个 LLM 并行生成多份不同的食谱，然后将这些结果汇总起来生成一份购物清单。**

3. **Writing Stories(故事创作)**

**通过将问题拆分成多个组成部分，利用 LLM 来创作书籍或故事。例如，可以先撰写故事摘要，然后塑造人物角色、构建故事情节节点，最后再进一步创作具体的对话**

# 四、使用生成式模型进行推理

在之前的模块这种，我们更关注提示词的模块化组成，并通过迭代来构架它们。这些该的提示词工程技术，比如说提示词链，被证实是让生成式模型进行复杂推理的第一步。

Reasoning 是人类智能的核心组件，并且它经常和 LLM 的新兴行为进行比较，这种行为**非常像（resemble）**是推理。我们强调*"resemble"*，因为，在撰写本书时，人们普遍认为模型通过对训练数据的记忆和模式匹配来展示它们的行为。

然而，这些 LLM 生成的输出也可以展示复杂的行为，即使那不是真正的推理，然而人们还是将其认定为具有推理能力。换句话说，我们通过提示词工程和 LLM 进行对话是为了模拟推理的过程以产生更好的输出。

为了简单的理解 LLM 的推理过程，我们将推理的方式分为两个系统。**系统 A：** 的思维那些自动的、直觉性的且几乎瞬间的过程；它和生成式模型不用自己反思地自动生成 token 非常相似。相反，**系统 B：** 的思维是连续的、缓慢的且有逻辑的过程，就像头脑风暴和自我反思一样。


在这个部分，我们将探索几种技术来模拟人类的思考过程，目的是改善模型的输出

## 4.1 CoT（Chain-of-Thought）: 在回答之前思考

让生成式模型进行推理的第一步且重要的步骤是**CoT(Chain-of-thought)**。**CoT 的目标是让生成式模型先进行“思考”，而不是没有经过任何推理后直接回答问题**。这种方法对于数学问题复杂程度较高的任务尤其有帮助。如下图所示，我们可以看到，我们在提供问题的答案时不再直接使用最终步骤的答案，而是增加了一个循序渐进的过程来计算出最终步骤的答案

<center>
<img src="./resources/CoT_prompt.png">
</center>



In [16]:
cot_prompt = [
    {"role": "user", "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"},
    {"role": "assistant", "content": "Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."},
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"}
]

outputs = pipe(cot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The cafeteria started with 23 apples. They used 20 apples for lunch, so they had 23 - 20 = 3 apples left. After buying 6 more apples, they now have 3 + 6 = 9 apples. The answer is 9.


与提供例子相反，我们可以使用一种简单的询问方式来让生成式模型提供推理过程（**Zero-Shot chain-of-thought**）。有很多种不同的方式可以实现，但是**最常见的、也是最有效的方式是使用“Let's think step-by-step”（让我们来一步一步的思考吧）**，如下图所示

<center>
<img src="./resources/CoT_prompt_zero_shot.png">
</center>

In [17]:
# 零样本的提示词
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]

outputs = pipe(zeroshot_cot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Start with the initial number of apples in the cafeteria, which is 23.

Step 2: Subtract the number of apples used to make lunch, which is 20.
23 - 20 = 3 apples remaining.

Step 3: Add the number of apples bought, which is 6.
3 + 6 = 9 apples.

So, the cafeteria now has 9 apples.


看！我们没有提供示例，它也能得到相同的推理行为。

**需要注意一点的是：“Let's think step by step” 并不是万用的方式，与它相似的方式还可以是“Take a deep breath and think step by step” 或者 “Let's work through the problem step by step.”**

## 4.2 Self-Consistency(自洽性)：样例输出

一个提示词调用多次 LLM 可能生成的结果也会不同。因此，为了抵消这种随机性的影响且改善生成式模型的性能，我们引入了**自洽性（self-consistency）：该方法会使用相同的提示词调用生成式模型多次，随后它去选择结果中最多的那个成为最终的答案
。** 然而，这种方式会调用多次生成式模型，系统的性能会慢 N 倍，如果他调用了 N 次的话。具体的流程如下图所示：

<center>
<img src="./resources/self_consistency_prompt.png">
</center>

## 4.3 Tree-of-Thought（ToT, 思维树）: 探索内部的步骤

CoT 和 Self-consistency 只是当前用于模拟复杂推理的众多方法中的一小部分，在这些方法的基础上，**思维树（Tree-of-Thought）**提供了一种改进方案，使模型能够对多种思路进行深入探索。如下图所示，我们通过提示词引导生成式模型探索当前问题的不同方案，随后，模型通过投票选出最佳方案，并继续执行下一个步骤

<center>
<img src="./resources/tree-of-thought.png">
</center>

在需要多步思考的时候此方法非常有用，就比如说写一个故事或者想一个有创意的想法的时候。但是，它也有一个缺点，需要调多次生成式模型，性能会慢很多。幸运的是，我们可以使用提示词来模拟这个框架(ToT)，而不是去调用大模型多次。**其主要的方式为：通过一段 Prompt, 让单个大模型模拟多位专家相互讨论、自我纠错的过程。** 中英文提示词模板如下：

A. 中文提示词：

```text
假设有三位不同的专家共同回答以下问题。

每个专家分别提出自己的分析，然后与其他专家分享。

接下来，所有专家继续分析问题，互相检查其他专家的观点。

如果某位专家发现自己的观点存在错误，就应该承认错误并退出当前讨论。

最终，通过讨论形成一个共同答案。

问题：
{{question}}
```

B. 英文提示词：

```text
Imagine three different experts are answering this question.

All experts will write down 1 step of their thinking, then share it with the group.

Then all experts will go on to the next step, etc. If any expert realizes they're wrong at any point then they leave.

The question is {{question}}

Make sure to discuss the results.

```

In [18]:
# 零样本的 ToT
zeroshot_tot_prompt = [
    {"role": "user", "content": "Imagine three different experts are answering this question. All experts will write down 1 step of their thinking, then share it with the group. Then all experts will go on to the next step, etc. If any expert realizes they're wrong at any point then they leave. The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?' Make sure to discuss the results."}
]

outputs = pipe(zeroshot_tot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Expert 1:
Step 1: Start with the initial number of apples, which is 23.

Expert 2:
Step 1: Subtract the number of apples used for lunch, which is 20.
Step 2: Add the number of apples bought, which is 6.

Expert 3:
Step 1: Start with the initial number of apples, which is 23.
Step 2: Subtract the number of apples used for lunch, which is 20.
Step 3: Add the number of apples bought, which is 6.

Results:
All three experts arrived at the same answer:

Expert 1: 23 - 20 + 6 = 9 apples
Expert 2: (23 - 20) + 6 = 9 apples
Expert 3: (23 - 20) + 6 = 9 apples

All three experts agree that the cafeteria has 9 apples left.


# 五、输出验证

最终，用生成式模型构建的系统和应用可能会部署到生产环境中，当这种情况发生的时候，我们验证和控制模型的输出就变得非常重要，为了让应用更好的运行，也为了创建一个健壮的生成式的 AI 系统。

验证输出的原因可能包括如下几种：

1. *Structured output*

    默认情况下，大多数的生成式模型会生成没有特定格式的自由文本。但一些特定的情况下，是需要让生成式模型生成固定的输出的，比如 JSON

2. *Valid output*

    即使我们要求模型生成结构化输出的能力，它仍然有能力去随意的生成它的内容。

3. *Ethics*

    一些开源的生成式模型没有安全围栏，那么它将会不会考虑安全和伦理来生成内容。举个例子，如果用户要求以下这些内容输出时，我们应该做拦截

- profanity: 脏话
- personally identifiable information(PII): 个人身份信息
- bias: 偏见
- cultural stereotypes: 文化刻板印象
- 等等

4. *Accuracy*

    生成的文本尽可能准确，连贯且少幻觉

通常情况下，有三种方式控制生成式模型的输出

1. Exmaples

给大模型提供预期输出的实例

2. Grammar

控制 Token 选择的过程

3. Fine-tuning

在数据上微调一个模型使其包含预期的输出


下面，我们详细说前两个方法，微调的技术在后续章节会讲述

## 5.1 提供示例

In [19]:
# 零样本学习
zeroshot_prompt = [
    {"role": "user", "content": "Create a character profile for an RPG game in JSON format."}
]

# Generate the output
outputs = pipe(zeroshot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "name": "Aria Stormbringer",
  "class": "Warrior",
  "race": "Human",
  "level": 10,
  "attributes": {
    "strength": 18,
    "dexterity": 12,
    "constitution": 16,
    "intelligence": 8,
    "wisdom": 10,
    "charisma": 14
  },
  "skills": {
    "melee": 15,
    "ranged": 10,
    "magic": 5,
    "stealth": 12,
    "acrobatics": 10,
    "animal_handling": 8
  },
  "equipment": {
    "weapon": "Two-handed Axe",
    "armor": "Chainmail Hauberk",
    "shield": "Warhammer",
    "accessories": [
      "Warrior's Talisman",
      "Leather Boots",
      "Woolen Cloak"
    ]
  },
  "background": "Aria grew up in a small village on the outskirts of a great city. She was always fascinated by the stories of brave warriors who fought to protect their people. When she was just a child, her village was attacked by bandits, and she witnessed the bravery of a local warrior who saved her family. From that day on, Aria knew that she wanted to become a warrior herself. She trained hard an

In [20]:
# 单样本学习
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:

{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [
    {"role": "user", "content": one_shot_template}
]

# 生成输出
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "description": "A cunning rogue with a mysterious past, skilled in stealth and deception.",
  "name": "Shadowcloak",
  "armor": "Leather Hood",
  "weapon": "Dagger"
}


**在这里注意一点，模型是否遵循你提供的格式仍然取决于模型本身。一些模型本就不擅长于遵循指令**

## 5.2 语法：约束后的样例

少样本学习有一个很大的**缺点**：**我们无法明确阻止某些输出被生成。** 虽然我们可以引导模型并向它提供指令，但模型仍然可能不会完全遵循这些要求。因此，人们迅速开发出了许多用于约束和验证生成式模型输出的工具包，例如 Guidance、Guardails 和 LMQL。

1. 我们可以让 LLM 做自身验证

<center>
<img src="./resources/model_own_validate.png">
</center>

2. 我们还可以让模型做填空题

<center>
<img src="./resources/cloze.png">
</center>

3. 更进一步，我们可以直接在 **token 采样阶段** 进行约束和验证，例如情感分析

<center>
<img src="./resources/constrain_selection.png">
</center>



让我们用 `llama-cpp-python` 来解释这些现象，它也是一个库，类似于 `transformer`，我们可以用它来加载模型。它通常用于高效的加载和使用压缩的模型。

在 `llama-cpp-python` 中，模型存储的格式为 GGUF。GGUF 文件里不只是模型权重，还会一起保存很多元数据，例如模型架构、tensor 信息、量化类型、tokenizer 相关信息、上下文参数等等。官方实现中，GGUF 大体包含文件头，key-value 元数据、tensor 描述以及真正的 tensor 数据。

**注意：实际开发中，当一个 notebook 中模型改变了之后，需要更换 notebook 再运行**。不然，会产生 OOM 等不利的影响

在这里，我使用代码来清空 GPU 的显存（VRAM），来模拟更换 notebook 的过程

In [21]:
import gc
import torch

del model, tokenizer, pipe

gc.collect()
torch.cuda.empty_cache()

现在，我们已经清空内存了。我们依旧加载 `Phi-3`。

在下面的代码中，设置 `n_gpu_layers = 1` 是为了让模型的所有层都使用上 GPU 加速。`n_ctx` 代表模型的上下文窗口的大小。`repo_id` 和 `filename` 的选择可以参考 HuggingFace 库中[
Phi-3-mini-4k-instruct](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf) 模型的说明

In [1]:
%pip install -U llama-cpp-python

In [2]:
from llama_cpp.llama import Llama

# 加载 Phi-3
llm = Llama.from_pretrained(
    repo_id="microsoft/Phi-3-mini-4k-instruct-gguf",
    filename="*fp16.gguf",
    n_gpu_layers=-1,
    n_ctx=2048,
    verbose=False
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./Phi-3-mini-4k-instruct-fp16.gguf: reconstructing file:   0%|          |  0.00B / 7.64GB            

./Phi-3-mini-4k-instruct-fp16.gguf: downloading bytes:           |  0.00B            

为了使用内部的 JSON Grammer 来生成输出。我们需要指定 `response_format` 来作为 JSON 对象。在内部，它会使用 JSON Grammar 来确保输出遵循 JSON 格式。

In [3]:
output = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "Create a warrior for an RPG in JSON format."}
    ],
    response_format={"type": "json_object"},
    temperature=0
)['choices'][0]['message']['content']

我们来检验输出是否是 JSON 格式

In [4]:
import json

output_str = json.dumps(output, indent=4)
print(output_str)

"{\n  \"warrior\": {\n    \"name\": \"Eldric Stormbringer\",\n    \"class\": \"Warrior\",\n    \"level\": 5,\n    \"attributes\": {\n      \"strength\": 18,\n      \"dexterity\": 10,\n      \"constitution\": 16,\n      \"intelligence\": 8,\n      \"wisdom\": 10,\n      \"charisma\": 12\n    },\n    \"skills\": [\n      {\n        \"name\": \"Martial Arts\",\n        \"proficiency\": 20,\n        \"description\": \"Expert in hand-to-hand combat and weapon handling.\"\n      },\n      {\n        \"name\": \"Shield Block\",\n        \"proficiency\": 18,\n        \"description\": \"Highly skilled at deflecting attacks with a shield.\"\n      },\n      {\n        \"name\": \"Heavy Armor\",\n        \"proficiency\": 16,\n        \"description\": \"Expertly equipped with heavy armor for protection.\"\n      },\n      {\n        \"name\": \"Survival\",\n        \"proficiency\": 14,\n        \"description\": \"Adept at finding food, water, and shelter in the wilderness.\"\n      }\n    ],\n    

输出的结果是 JSON，由于显示的原因，我给一种更清晰的显示

```json
{
  "warrior": {
    "name": "Eldric Stormbringer",
    "class": "Warrior",
    "level": 5,
    "attributes": {
      "strength": 18,
      "dexterity": 10,
      "constitution": 16,
      "intelligence": 8,
      "wisdom": 10,
      "charisma": 12
    },
    "skills": [
      {
        "name": "Martial Arts",
        "proficiency": 20,
        "description": "Expert in hand-to-hand combat and weapon handling."
      },
      {
        "name": "Shield Block",
        "proficiency": 18,
        "description": "Highly skilled at deflecting attacks with a shield."
      },
      {
        "name": "Heavy Armor",
        "proficiency": 16,
        "description": "Expertly equipped with heavy armor for protection."
      },
      {
        "name": "Survival",
        "proficiency": 14,
        "description": "Adept at finding food, water, and shelter in the wilderness."
      }
    ],
    "equipment": [
      {
        "name": "Iron Sword",
        "type": "Weapon",
        "damage": 12,
        "durability": 100
      },
      {
        "name": "Chainmail Armor",
        "type": "Armor",
        "defense": 18,
        "durability": 100
      },
      {
        "name": "Leather Boots",
        "type": "Armor",
        "defense": 8,
        "durability": 100
      }
    ],
    "abilities": [
      {
        "name": "Berserker Rage",
        "description": "Increases strength and attack power for a short duration."
      },
      {
        "name": "Shield Wall",
        "description": "Forms a defensive barrier with allies, reducing incoming damage."
      },
      {
        "name": "Battle Cry",
        "description": "Inspires nearby allies, increasing their attack power temporarily."
      }
    ]
  }
}
```